# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a practical step-by-step guide for loading and exploring the FAIRˆ² dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and print basic metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets and their fields. Use `@id` to reference every entity for reproducibility and clarity.

Print Record Set `@id`s and available Field `@id`s and types.

In [ ]:
# List all record sets by @id
record_sets = dataset.metadata.record_sets
print("Available Record Sets and their Fields:")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}")
    if 'fields' in rs:
        for f in rs['fields']:
            field_id = f.get('@id', None)
            name = f.get('name', '')
            dtype = f.get('dataType', '')
            print(f"    - Field @id: {field_id}, name: {name}, dataType: {dtype}")


## 3. Data Extraction

Load data from a specific Record Set into a dataframe for analysis.

Replace `<record_set_id>` with a valid Record Set `@id` from above. Use field and column `@id` as needed.

In [ ]:
# Define the list of record set @id's available in this dataset
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set into a DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# For demonstration, choose the first Record Set
main_record_set_id = record_set_ids[0]
print(f"Fields/columns in record set {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")

# Display a preview
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on a numeric field, normalization, and grouping.

Make sure to use only field or column `@id` for references.

In [ ]:
# Select a numeric field for analysis
# Inspect columns for numeric fields - this example assumes a column '@id' is used for numeric values, adjust as needed
df = dataframes[main_record_set_id]
numeric_cols = df.select_dtypes(include='number').columns.tolist()
if not numeric_cols:
    raise Exception("No numeric columns found for EDA.")
numeric_field_id = numeric_cols[0]
print(f"Using numeric field '@id': {numeric_field_id}")

# Filtering
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'O' else 10  # fallback
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (using field @id):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally group by a non-numeric field
non_numeric_cols = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
if non_numeric_cols:
    group_field_id = non_numeric_cols[0]
    print(f"Grouping by '@id': {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id} (showing first few groups):")
    print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. We will plot the distribution of the selected numeric field and compare group means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping was performed, plot group means
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    grouped_df[numeric_field_id].plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR⁲ dataset using the `mlcroissant` library, explored its record sets via `@id`, extracted key fields with their `@id`s, and performed basic EDA by filtering and visualizing a chosen numeric field. This workflow can be extended for more advanced processing or analysis tasks relevant to your scientific or analytical goals.